# Checkpoint Week 1 — KNN

Vul alle cellen in en push dit bestand naar je repo. De automatische checks (GitHub Actions) voeren dit notebook uit en controleren of alle **variabelen met de gevraagde namen** bestaan en correct zijn.

**BELANGRIJK**: gebruik exact de genoemde variabelennamen (`accuracy_iris`, `mse_mall`, ...), anders faalt de controle.

## Oefening 1 — Eigen KNN-implementatie

Implementeer `KNNClassifier`. De constructor krijgt `k` mee; `fit` bewaart de trainingsdata; `predict` geeft voor elk punt het meerderheidslabel van de k dichtste buren (Euclidische afstand).

In [ ]:
import numpy as np

class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for x in X:
            dists = np.sqrt(((self.X_train - x) ** 2).sum(axis=1))
            idx = np.argsort(dists)[:self.k]
            labels, counts = np.unique(self.y_train[idx], return_counts=True)
            preds.append(labels[np.argmax(counts)])
        return np.array(preds)

## Oefening 2 — Iris met sklearn-KNN

- Laad de iris-dataset, splits in train/test (test_size=0.25, random_state=42).
- **Normaliseer** met een StandardScaler (fit op train alleen!).
- Train een `KNeighborsClassifier(n_neighbors=5)`.
- Sla de voorspellingen op in `y_pred_iris` en de **accuracy op de testset** in `accuracy_iris` (float).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
y_pred_iris = knn.predict(X_test_scaled)
accuracy_iris = accuracy_score(y_test, y_pred_iris)
print(accuracy_iris)

**Vraag (antwoord in `antwoord_normaliseren`)**: Waarom moet je de features normaliseren vóór KNN? Kies A, B, C of D:
- A: Omdat KNN alleen met integers werkt
- B: Omdat afstanden anders vertekend worden door schaalverschillen tussen features
- C: Omdat accuracy dan altijd 100% wordt
- D: Omdat sklearn dat vereist voor alle modellen

In [ ]:
antwoord_normaliseren = "B"

## Oefening 3 — Mall Customers (one-hot + regressie)

- Laad `data/Mall_Customers.csv` (staat in dezelfde map als dit notebook).
- One-hot encode `Gender`, voeg samen met `Age` en `Annual Income (k$)`.
- Doelvariabele: `Spending Score (1-100)` → gebruik een `KNeighborsRegressor(n_neighbors=5)` met train/test split (test_size=0.25, random_state=42).
- Sla de test-voorspellingen op in `y_pred_mall` en de **MSE** in `mse_mall` (float).

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('data/Mall_Customers.csv')
enc = OneHotEncoder(drop='first', sparse_output=False)
gender_enc = enc.fit_transform(df[['Gender']])
X_mall = pd.DataFrame(gender_enc, columns=['Gender_encoded'])
X_mall[['Age', 'Annual Income (k$)']] = df[['Age', 'Annual Income (k$)']].values
y = df['Spending Score (1-100)']
X_train, X_test, y_train, y_test = train_test_split(X_mall, y, test_size=0.25, random_state=42)
reg = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
y_pred_mall = reg.predict(X_test)
mse_mall = mean_squared_error(y_test, y_pred_mall)
print(mse_mall)

**Vraag (antwoord in `antwoord_k_kiezen`)**: Hoe kies je K het best? Kies A, B, C of D:
- A: Altijd K=1
- B: Zomaar op het gevoel
- C: Via cross-validatie
- D: K moet altijd gelijk zijn aan het aantal klassen

In [ ]:
antwoord_k_kiezen = "C"